# 08 · Agent framework client → Bifrost (LLM + MCP tools)
The finale: a **Microsoft Agent Framework** agent whose *model calls* go through the gateway's OpenAI endpoint and whose *tools* come from the S.H.I.E.L.D. MCP server – also through APIM. One key, one audit trail.

> Agent Framework is evolving quickly; if an import/signature differs, see https://learn.microsoft.com/agent-framework. The gateway part (an OpenAI-compatible client with `base_url` + `api-key`) is framework-agnostic – LangChain's `ChatOpenAI(base_url=..., default_headers=...)` works identically.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

from openai import AsyncOpenAI
from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework.openai import OpenAIChatCompletionClient

key = user_key('tony-stark')
llm = OpenAIChatCompletionClient(model='heimdal-auto',
                       async_client=AsyncOpenAI(base_url=env('HEIMDAL_LLM_BASE_URL'), api_key=key, default_headers={'api-key': key, 'x-agent': 'friday'}))
shield = MCPStreamableHTTPTool(name='shield', url=env('HEIMDAL_MCP_URL'), static_headers={'api-key': key, 'x-agent': 'friday'})

async with Agent(client=llm, name='Friday', tools=shield, default_options={'reasoning_effort': 'none'},
                     instructions='You are F.R.I.D.A.Y. Use the S.H.I.E.L.D. tools for facts about the Avengers, missions and Infinity Stones.') as agent:
    r = await agent.run('Which Avengers are available, and where is the Soul Stone?')
    print(r.text)

Finally, see what Friday cost – the gateway tags every call with the `x-agent` header, so it shows up as its own `Agent` (metrics land in App Insights, allow a few minutes):

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

kql_wait(KQL_TOKENS_BY_AGENT_METRIC, hours=3)